# 03 — International-only split and train-fitted preprocessing

**No new split is created.** We restrict the original global official train/test membership to the explicitly named international cohort, preventing leakage between the existing global and U.S. experiments. The **test labels are saved only for a future final evaluation**, not inspected for model selection in `04` or `05`.

Base preprocessing below is an artifact for parity with the U.S. notebook. All actual predictive models in `05/06` must use a full fold-fitted pipeline from `utils/international_features.py`. Country/region is used instead of the incorrect U.S. `state` variable.

In [1]:
from pathlib import Path
import sys
HERE = Path.cwd()
PROJECT_ROOT = next((folder for folder in (HERE, *HERE.parents)
                     if (folder / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the 1C repository")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
GLOBAL_DIR = PROJECT_ROOT / "data" / "Processed-Data"
INT_DIR = GLOBAL_DIR / "international_only"
CLEAN_GLOBAL = GLOBAL_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from utils.international_scope import align_international_to_global_split, classify_locations, international_mask
from utils.preprocessing import drop_leaks
paths = [INT_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv",
         GLOBAL_DIR / "train_project_ids.csv", GLOBAL_DIR / "test_project_ids.csv"]
for path in paths:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}; run international 02 and global 03 first")
clean = pd.read_csv(paths[0])
global_train_ids = pd.read_csv(paths[1])
global_test_ids = pd.read_csv(paths[2])
assert clean["project id"].is_unique and international_mask(clean).all()
train_ids, test_ids = align_international_to_global_split(clean, global_train_ids, global_test_ids)
print("International training projects:", len(train_ids))
print("International test projects (reserved):", len(test_ids))
print("Training success rate:", round(train_ids["target"].mean(), 4) if len(train_ids) else "n/a")
assert set(train_ids["project id"]).isdisjoint(test_ids["project id"])
if len(train_ids) < 100 or len(test_ids) < 20:
    print("WARNING: very small labeled cohorts; interpret future results cautiously")

International training projects: 1427
International test projects (reserved): 637
Training success rate: 0.6097


In [2]:
by_id = clean.set_index("project id", drop=False)
train_rows = by_id.loc[train_ids["project id"]].reset_index(drop=True)
test_rows = by_id.loc[test_ids["project id"]].reset_index(drop=True)
assert len(train_rows) + len(test_rows) == clean["status"].isin(["successful", "failed"]).sum()
assert np.array_equal(train_rows["status"].eq("successful").astype(int), train_ids["target"].to_numpy())
assert np.array_equal(test_rows["status"].eq("successful").astype(int), test_ids["target"].to_numpy())
X_train, X_test = drop_leaks(train_rows.drop(columns="status")), drop_leaks(test_rows.drop(columns="status"))
train_geo = classify_locations(X_train["location"])
test_geo = classify_locations(X_test["location"])
assert train_geo["scope"].eq("explicit_non_us").all()
assert test_geo["scope"].eq("explicit_non_us").all()

# Ensure no overlap with the saved U.S. split, if it is present.
us_tr = GLOBAL_DIR / "us_only" / "train_project_ids.csv"
us_te = GLOBAL_DIR / "us_only" / "test_project_ids.csv"
if us_tr.exists() and us_te.exists():
    us_ids = set(pd.read_csv(us_tr)["project id"]) | set(pd.read_csv(us_te)["project id"])
    assert not (us_ids & set(train_ids["project id"]) | us_ids & set(test_ids["project id"]))
    print("Confirmed U.S. and named-international modeling project IDs are disjoint")
print("Countries represented in training:", int(train_geo["country_region"].nunique()))
print("Training class counts:", train_ids["target"].value_counts().to_dict())

Confirmed U.S. and named-international modeling project IDs are disjoint
Countries represented in training: 125
Training class counts: {1: 870, 0: 557}


In [3]:
# Raw categorical country evidence is explicitly named; do not masquerade as a U.S. state.
base_train = X_train[["goal", "levels", "duration", "category", "subcategory"]].copy()
base_test = X_test[["goal", "levels", "duration", "category", "subcategory"]].copy()
base_train["country_region"] = train_geo["country_region"].to_numpy()
base_test["country_region"] = test_geo["country_region"].to_numpy()
NUMERIC = ["goal", "levels", "duration"]
CATEGORICAL = ["category", "subcategory", "country_region"]
base_preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAL),
], verbose_feature_names_out=True)
train_base = pd.DataFrame(base_preprocessor.fit_transform(base_train), columns=base_preprocessor.get_feature_names_out())
test_base = pd.DataFrame(base_preprocessor.transform(base_test), columns=base_preprocessor.get_feature_names_out())
assert train_base.shape[1] == test_base.shape[1]
print("Train-fitted preprocessing shapes:", train_base.shape, test_base.shape)
print("Unknown test categories are safely ignored by OneHotEncoder; do not evaluate test outcomes here")

Train-fitted preprocessing shapes: (1427, 190) (637, 190)
Unknown test categories are safely ignored by OneHotEncoder; do not evaluate test outcomes here


In [4]:
INT_DIR.mkdir(exist_ok=True, parents=True)
train_ids.to_csv(INT_DIR / "train_project_ids.csv", index=False)
test_ids.to_csv(INT_DIR / "test_project_ids.csv", index=False)
pd.DataFrame({"target": train_ids["target"]}).to_csv(INT_DIR / "y_train.csv", index=False)
pd.DataFrame({"target": test_ids["target"]}).to_csv(INT_DIR / "y_test.csv", index=False)
train_base.to_csv(INT_DIR / "X_train_base_preprocessed.csv", index=False)
test_base.to_csv(INT_DIR / "X_test_base_preprocessed.csv", index=False)
joblib.dump(base_preprocessor, INT_DIR / "base_preprocessor.joblib")
print("Saved seven international split/base-preprocessing artifacts to", INT_DIR)
print("These do not overwrite the U.S. or global processed data")

Saved seven international split/base-preprocessing artifacts to /Users/annasehgal/State-Street-1C-kickstarter-crowdfunding-recommendation-engine/data/Processed-Data/international_only
These do not overwrite the U.S. or global processed data


**Next:** `04_feature_engineering.ipynb` performs exploratory outcome analysis **only on international training rows**, then inspects international-specific conservative features.